# Lesson 7, Part 4

In [ ]:
# initializations and helper functions from earlier parts
import numpy as np
import matplotlib.pyplot as plt

def inv_sin(x, tol=1e-15):
    theta = np.pi / 4
    diverge = 1 / np.sqrt(2) - x
    while abs(diverge) > tol:
        theta -= diverge / reduced_cosine(theta)
        diverge = reduced_sine(theta) - x
    return theta

def taylor_cosine(theta, order=40):
    theta_square = theta * theta
    cosine = next_term = 1
    for k in np.arange(2, order + 1, 2):
        next_term *= theta_square / (k * (1 - k))
        cosine += next_term
    return cosine

def polar_unlabeled():
    plt.axes(projection='polar')
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    plt.box([])

def taylor_sine(theta, order=40):
    theta_square = theta * theta
    if np.isscalar(theta):
        next_term = sine = theta
    else:
        next_term = theta.copy()
        sine = theta.copy()
    for k in np.arange(3, order + 1, 2):
        next_term *= theta_square / (k * (1 - k))
        sine += next_term
    return sine

def reduced_sine(theta):
    theta = theta % (2 * np.pi)
    switch_sign = 1
    if theta > np.pi:
        switch_sign = -1
        theta = 2 * np.pi - theta
    if theta > np.pi / 2:
        theta = np.pi - theta
    if theta > np.pi / 4:
        return switch_sign * taylor_cosine(np.pi / 2 - theta, order=16)
    return switch_sign * taylor_sine(theta, order=17)

def reduced_cosine(theta):
    theta = theta % (2 * np.pi)
    if theta > np.pi:
        theta = 2 * np.pi - theta
    switch_sign = 1
    if theta > np.pi / 2:
        switch_sign = -1
        theta = np.pi - theta
    if theta > np.pi / 4:
        return switch_sign * taylor_sine(np.pi / 2 - theta, order=17)
    return switch_sign * taylor_cosine(theta, order=16)

Let's graph sin<sup>$m$</sup>$ \theta$  cos<sup>*n*</sup>$ \theta$ for various nonnegative integers *m* and *n*.

In [ ]:
def sin_cos_powers(m,n,theta):
    return np.sin(theta)**m * np.cos(theta)**n

theta = np.linspace(-np.pi,np.pi,400)
plt.style.use('seaborn')
for m in (0,1,2,3):
    plt.figure()
    plt.title('Power '+str(m)+' of Sine times Power n of Cosine',)
    for n in (1,2,3,4):
        y = sin_cos_powers(m,n,theta)
        plt.plot(theta,y,label='n ='+str(n))
    plt.legend()
plt.show()

Other powers of sines and cosines can be integrated only on regions where they are well-defined. 
They are well-defined for all $\theta$ only if the powers are fractions of odd whole numbers. 
For tiny powers they approach step functions flipping between -1 and +1.

In [ ]:
theta = np.linspace(-np.pi,np.pi,400)
plt.style.use('seaborn')
plt.title('Fractional Powers of Sine')
for m in (1,3,5,7):
    sine = np.sin(theta)
    y = np.sign(sine) * np.abs(sine)**(1/m)
    plt.plot(theta,y,label='power = 1/'+str(m))
plt.legend()
plt.show()

Let's integrate *f*($\theta$) = sin<sup>*p*</sup> $\theta$ cos<sup>2*n*+1</sup> $\theta$ for nonnegative integer *n*. 
As explained in the text, substituting *x* = sin $\theta$ converts *f*($\theta$) *d*$\theta$ into 
*x*<sup>*p*</sup>  (1-*x*<sup>2</sup>)<sup>*n*</sup> *dx*. We then expand and integrate. Note that *x* = 0 when $\theta$ = 0.

In [ ]:
def integral_curious_fn(p,n,x):
    # function is x**p (1-x)**(2n+1)
    neg_x_square = -(x * x)
    x_power = combin = 1
    integral = 1 / (p + 1)
    for k in range(1,n+1):
        combin *= (n - k + 1) / k
        x_power *= neg_x_square
        integral +=  x_power * combin / (2 * k + p + 1)
    integral *= x**(p+1)
    return integral

def integral_sin_p_cos_odd(p,n,theta):
    return integral_curious_fn(p,n,np.sin(theta))
    
p,n = 2,1
theta = np.linspace(-np.pi,np.pi,400)
y = sin_cos_powers(p,2*n+1,theta)
integral = integral_sin_p_cos_odd(p,n,theta)
plt.style.use('seaborn-v0_8')
plt.title('y = sin(x)**'+str(p)+' * cos(x)**'+str(2*n+1)+' and its integral')
plt.plot(theta,y,theta,integral)
plt.legend(labels=('y','integral of y'))
plt.show()

Let's check these results using a simplied version of deriv_approx from Level 6 Part 3 
and the refined counterpart **gradient** built into numpy.

In [ ]:
def deriv_approx(y,dx):
    dy = np.zeros(len(y)+1)
    dy[1:-1] = np.diff(y)
    dy[0] = dy[1]
    dy[-1] = dy[-2]
    return (dy[:-1] + dy[1:]) / (2 * dx)

def report_divergence(divergence,theta):  
    plt.plot(theta,divergence)
    max_diverge = np.max(np.abs(divergence))
    avg_diverge = np.sum(np.abs(divergence) / len(divergence))
    print('Absolute divergence averages','{:.0e}'.format(avg_diverge),'with maximum','{:.1e}'.format(max_diverge))

p,n = 2,1
theta = np.linspace(-np.pi,np.pi,400)
y = sin_cos_powers(p,2*n+1,theta)
integral = integral_sin_p_cos_odd(p,n,theta)
dtheta = theta[1] - theta[0]
print('Using deriv_approx:')
diverge1 = deriv_approx(integral,dtheta) - y
report_divergence(diverge1,theta)
print('Using np.gradient:')
diverge2 = np.gradient(integral,dtheta) - y
report_divergence(diverge2,theta)
plt.title('Divergence between deriv(integral(y)) and y')
plt.show()

Let's do similar calculations for the integral of sin<sup>2*m*+1</sup> $\theta$ cos<sup>*p*</sup> $\theta$ or nonnegative integer *m*. 
In this case, the definite integral from $\theta$ = 0 must subtract a non-zero value from the indefinite integral.

In [ ]:
def integral_sin_odd_cos_p(m,p,theta):
    integral = -integral_curious_fn(p,m,np.cos(theta))
    integral_0 = -integral_curious_fn(p,m,1)
    return integral - integral_0

m,p = 2,1
theta = np.linspace(-np.pi,np.pi,400)
y = sin_cos_powers(2*m+1,p,theta)
integral = integral_sin_odd_cos_p(m,p,theta)
plt.title('Integral of y = sin**'+str(2*m+1)+' * cos**'+str(p))
plt.plot(theta,y,theta,integral)
plt.legend(labels=('y','Integral of y'))
plt.figure()
plt.title('Divergence between deriv(integral(y)) and y')
deriv = np.gradient(integral,theta[1]-theta[0])
plt.plot(theta,deriv-y,label='sin_odd_cos_p')
plt.legend()
plt.show()

The text derived an exact integral for sqrt(1-*x*<sup>2</sup>). 
Let's check this using functions from Level 6 Part 2 and an exact formula that uses either inv_sin or the numpy function **arcsin**.

In [ ]:
def integral_approx(y,dx):
    avg_y = 0.5 * (y[:-1] + y[1:])
    integral = np.zeros(len(y))
    integral[1:] = np.cumsum(avg_y) * dx 
    return integral

def inv_sin(x,tol=1E-15):
    theta = np.pi/4
    diverge = 1/np.sqrt(2) - x
    while abs(diverge) > tol:
        theta -= diverge / taylor_cosine(theta)
        diverge = taylor_sine(theta) - x # cos(pi/4-theta) = since(theta)
    return theta   

num = 500
x = np.linspace(0,1,num)
y = np.sqrt(1 - x**2)
integral = integral_approx(y,x[1]-x[0])
plt.title('Divergence between integral(sqrt(1-x**2)) and exact answer')
print('Using inv_sin:')
exact1 = [0.5 * (inv_sin(x[i]) + x[i]*y[i]) for i in range(num)]
report_divergence(integral - exact1,x)
print('Using np.arcsin:')
exact2 = 0.5 * (np.arcsin(x) + x*y)
report_divergence(integral - exact2,x)
plt.show()

Let's do a similar comparison for the integral of 1/sqrt(1-*x*<sup>2</sup>). As usual, accuracy deteriorates when higher derivatives get very steep.

In [ ]:
num = 1000
x = np.linspace(0,0.999,num)
y = 1/np.sqrt(1 - x**2)
integral = integral_approx(y,x[1]-x[0])
plt.title('Divergence between integral(1/sqrt(1-x**2)) and exact answer')
print('Using inv_sin:')
exact1 = [inv_sin(val) for val in x]
report_divergence(integral - exact1,x)
print('Using np.arcsin:')
exact2 = np.arcsin(x)
report_divergence(integral - exact2,x)
plt.show()

Let's plot tan $\theta$ = sin $\theta$ / cos $\theta$ using **np.tan**. 
The raw display is dominated by the surges to huge positive values and flip to huge negative values. 
We need to bound the *y*-axis to see the intermediate pattern.

In [ ]:
theta = np.linspace(-np.pi,np.pi,1000)
y = np.tan(theta)
plt.subplot(2,1,1)
plt.title("Tangent of Angle")
plt.plot(theta,y)
plt.subplot(2,1,2)
plt.title("Close-up of Tangent")
plt.plot(theta,y)
plt.ylim(-4,4)  # confines display of y between -4 and +4
plt.xlabel('radians')
plt.show()

The slope of tan $\theta$ is 1 / (cos<sup>2</sup> $\theta$) = sec<sup>2</sup> $\theta$ = tan<sup>2</sup> $\theta$ + 1, 
which we will compare to the estimated gradient.

In [ ]:
theta = np.linspace(-0.85*np.pi/2,0.85*np.pi/2,1000)
y = np.tan(theta)
deriv_exact = 1 / np.cos(theta)**2
deriv_approx = np.gradient(y,theta[1]-theta[0])
plt.title('Derivative of Tangent of Angle')
plt.plot(theta,deriv_exact,theta,deriv_approx)
plt.legend(labels=('Exact','Approximate'))
plt.show()

The inverse tangent of *x* equals the integral of 1/(1+*x*<sup>2</sup>) from 0 to *x*, which we will estimate directly 
and compare to the numpy function **arctan**. The latter identifies angles in the central region (-$\pi$/2,$\pi$/2).

In [ ]:
num = 500
x = np.linspace(-6,6,num+1)
y = 1 / (x**2 + 1)
integral = integral_approx(y,x[1]-x[0])
integral -= integral[num//2]   # subtract value at x=0
exact = np.arctan(x)
plt.title('Divergence between integral(1/(x**2+1)) and np.arctan')
report_divergence(integral - exact,x)

plt.figure()
plt.title("Plot of 1/(x**2+1) and its integral from origin")
plt.plot(x,y,x,integral,x,exact)
plt.legend(labels=('1/(x**2+1)','integral','arctan'))
plt.axhline(y=0,linestyle=':')
plt.axvline(x=0,linestyle=':')
plt.show()

Integrating the Taylor series for 1/(1+*x*<sup>2</sup>) generates the Taylor series 
*x* - *x*<sup>3</sup> /3 + *x*<sup>5</sup>/5 - *x*<sup>7</sup>/7 + ... for arctan(*x*), which converges for |*x*| <= 1. 
For |*x*| > 1, we can substitute 1/*x* for *x* to calculate the inverse cotangent instead, and then subtract from $\pi$/2. 
However, as the code here shows, convergence is slow for |*x*| near 1.

In [ ]:
def taylor_arctan(x,tol=1E-14):
    sign_x = np.sign(x)
    x = np.abs(x)
    exceeds1 = x > 1
    if exceeds1:
        x = 1/x
    x_square = x**2
    arctan = x_power = 1 
    for order in range(3,200,2):
        x_power *= -x_square
        new_term = x_power / order
        arctan += new_term
        if np.abs(new_term) < tol:
            break
    arctan *= x
    if exceeds1:
        arctan = np.pi/2 - arctan
    arctan = np.sign(arctan) * arctan
    return arctan,order

num = 500
x = np.geomspace(0.01,100,num)
atan = np.zeros(num)
order = np.zeros(num)
for i in range(num):
    atan[i],order[i] = taylor_arctan(x[i])
atan_exact = np.arctan(x)
plt.title('Divergence between arctan and its Taylor series approximation')
report_divergence(atan - atan_exact,np.log(x))
plt.xlabel("Log of x")
plt.figure()
plt.title('Order of Taylor approximation')
plt.bar(np.log(x),order)
plt.xlabel("Log of x")
plt.show()

When $\theta$ = $\pi$/6 radians or 30 degrees, tan $\theta$ = sqrt(1/3). 
This implies $\pi$ = 6 arctan(sqrt(1/3)) = 2 sqrt(3) (1 - 1/(3&sdot;3) + 1/(5&sdot;3<sup>2</sup>) - 1/(7&sdot;3<sup>3</sup>) + ....). Let's compare our Taylor approximation to the stored **np.pi**.

In [ ]:
atan,order = taylor_arctan(1/np.sqrt(3))
pi_approx = 6 * atan
rel_diverge = np.log(pi_approx / np.pi)
print('The relative divergence between the Taylor series of order',order,'and the true π is','{:.0e}'.format(rel_diverge))

Let's model the flight of a hawk that maintains a constant pitch relative to prey, starting from distance *r*0 from the prey at angle 0. 
We'll define time in units of total distance traveled by the hawk, so that the measured velocity *v* = 1.

In [ ]:
def hawk_flight(pitch_in_degrees,r0,num=500):
    # set parameters for pitch and time
    pitch = pitch_in_degrees * np.pi/180  # = np.radians(pitch_in_degrees)
    sin_pitch = np.sin(pitch)
    cos_pitch = np.cos(pitch)
    maxtime = r0 / cos_pitch
    time = np.linspace(0,maxtime,num+1)
    dt = maxtime / num
    # prepare main loop
    theta = np.zeros(num+1)
    r = np.zeros(num+1)
    r[0] = r0
    for i in range(1,num):
        dr = -cos_pitch * dt
        r[i] = r[i-1] + dr
        avg_r = 0.5 * (r[i-1] + r[i])
        dtheta = sin_pitch * dt / avg_r
        theta[i] = theta[i-1] + dtheta       
    return time,r,theta

r0 = 100
plt.figure('Hawk Flights at Various Pitches')
polar_unlabeled()
for pitch_in_degrees in (15,30,45,60,75):
    time,r,theta = hawk_flight(pitch_in_degrees,r0)
    plt.plot(theta,r,label=str(pitch_in_degrees)+' deg')
plt.legend()
plt.show()

Plotted in terms of $\theta$ and ln *r* for *r* > 1, the paths are arithmetic spirals leading back to the origin.

In [ ]:
r0 = 100
plt.figure('Hawk Flights plotted in terms of log r')
polar_unlabeled()
for pitch_in_degrees in (15,30,45,60,75):
    time,r,theta = hawk_flight(pitch_in_degrees,r0)
    r[r<1]=1
    plt.plot(theta,np.log(r),label=str(pitch_in_degrees)+' deg')
plt.legend()
plt.show()

In theory, *r* = *r*0&sdot;exp(-$\theta$ cot $\psi$) = *r*0 - cos $\psi$ *vt*, 
which implies $\theta$ = -tan $\psi$  ln(1 - cos $\psi$ *vt* / *r*0). 
Let's plot the discrepancies for the computed values. While generally tiny, they widen as the spirals tighten.

In [ ]:
r0 = 100
for pitch_in_degrees in (15,30,45,60,75):
    time,r,theta = hawk_flight(pitch_in_degrees,r0)
    pitch = pitch_in_degrees * np.pi / 180
    time,theta = time[:-1],theta[:-1]  # trim series to avoid log(0) below
    theta_theory = -np.tan(pitch) * np.log(1 - np.cos(pitch) * time/r0)
    divergence = np.abs(theta - theta_theory)
    # if any divergence == 0, next line will generate error
    plt.semilogy(time,divergence,label=str(pitch_in_degrees)+' deg')
plt.title('Divergence between Theoretical and Computed θ')
plt.xlabel('Time')
plt.legend()
plt.show()